In [ ]:
import json, random, shutil
from pathlib import Path

# ---------- 参数 ----------
img_root   = Path('image') # 换成相应的图片文件夹路径
json_root  = Path('image') # 换成相应的标注文件夹路径
out_root   = Path('SplitDataset')
train_ratio, val_ratio = 0.8, 0.1
# ---------------------------

# 1. 创建目录结构
for sub in ['images/train','images/val','images/test',
            'labels/train','labels/val','labels/test']:
    (out_root/sub).mkdir(parents=True, exist_ok=True)

# 2. 随机划分
all_names = [p.stem for p in img_root.glob('*.jpg')]
random.shuffle(all_names)
n = len(all_names)
train_end = int(n*train_ratio)
val_end   = train_end + int(n*val_ratio)

splits = {
    'train': all_names[:train_end],
    'val'  : all_names[train_end:val_end],
    'test' : all_names[val_end:]
}

# 3. 类别映射（自行修改）
class2id = {'wfy':1, 'fy':2}

def json_to_yolo(json_path: str, txt_path: str):
    """
    将 labelme（或类似）标注的 json 转为 YOLO txt。
    - json_path : 输入的 .json 文件路径
    - txt_path  : 输出的 .txt 文件路径（同名、不同后缀）
    """
    # 读取 json
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    img_w = data["imageWidth"]
    img_h = data["imageHeight"]

    lines = []   # 每一行对应一个目标

    for shape in data.get("shapes", []):
        label = shape.get("label")
        if label not in class2id:
            # 若出现未知类别，直接跳过或自行添加映射
            continue

        points = shape.get("points", [])
        if not points:
            continue

        # ---------- 1️⃣ 计算外接矩形 ----------
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)

        # ---------- 2️⃣ 转为 YOLO 归一化格式 ----------
        x_center = (x_min + x_max) / 2.0 / img_w
        y_center = (y_min + y_max) / 2.0 / img_h
        w = (x_max - x_min) / img_w
        h = (y_max - y_min) / img_h

        # 防止极端情况（宽或高为 0）导致除零或无效标注
        if w <= 0 or h <= 0:
            continue

        class_id = class2id[label]
        line = f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}"
        lines.append(line)

    # ---------- 写入 txt ----------
    txt_path = Path(txt_path)
    txt_path.parent.mkdir(parents=True, exist_ok=True)
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

# 4. 按划分复制并转换
for split, names in splits.items():
    for name in names:
        # 复制图片
        src_img = img_root/f'{name}.jpg'
        dst_img = out_root/f'images/{split}/{name}.jpg'
        src_img = str(src_img).replace('\\', '/')
        dst_img = str(dst_img).replace('\\', '/')
        shutil.copy2(src_img, dst_img)

        # 转换标注
        json_path = json_root/f'{name}.json'
        json_path = str(json_path).replace('\\', '/')
        print('json_path', json_path)
        txt_path  = out_root/f'labels/{split}/{name}.txt'

        json_to_yolo(json_path, txt_path)

print("数据集划分完成！")